# Lesson 06 Lab — Diagnosing Coalescing with Strides

**Puzzle:** When offset geometry, transactions, and requested bandwidth change together, which observation tells you whether the kernel, layout, toolchain, or hardware boundary is responsible?

This notebook retains one complete RTX 5090 execution.


## Why this matters

This lab isolates offset geometry, transactions, and requested bandwidth and keeps its comparison path explicit.


## 0. Predict before running

Predict correctness, warm latency ordering, and the first boundary case. Write what would disprove each prediction.


## 1. Theory and mechanism

Coalescing follows the addresses requested by neighboring lanes, not the visual shape of a tensor. The same row-copy source can produce adjacent or strided addresses when its column stride changes. Requested bandwidth is useful only with the stride and byte formula beside it.


## 2. Trace the mechanism

```mermaid
flowchart LR
  A["Frozen input + contract"] --> B["offset geometry, transactions, and requested bandwidth"]
  B --> C["Triton candidate"]
  B --> D["CUDA / library control"]
  C --> E["correctness + samples"]
  D --> E
  E --> F["bounded decision"]
```


## 3. Inspect the comparison boundary

Baseline: named PyTorch CUDA/library or standard-grid path. Candidate: reviewed Triton kernel or explicit model described below.

Adding shared memory before proving an access problem can increase work without fixing the actual address pattern.


## 4. Inspect the execution environment

The next cell asserts CUDA and records GPU, target, PyTorch, CUDA runtime, Triton, Python, and seed.


In [1]:
from pathlib import Path
import json, sys

ROOT = Path.cwd().parents[2]
sys.path.insert(0, str(ROOT / "scripts"))
from chapter05_runtime import environment, run_lesson

LESSON_NO = 6
LESSON_TITLE = 'Diagnosing Coalescing with Strides'
ENV = environment(LESSON_NO)
print(json.dumps(ENV, indent=2, ensure_ascii=False))


{
  "gpu": "NVIDIA GeForce RTX 5090",
  "compute_capability": "12.0",
  "torch": "2.13.0+cu130",
  "cuda_runtime": "13.0",
  "triton": "3.7.1",
  "triton_target": "GPUTarget(backend='cuda', arch=120, warp_size=32)",
  "python": "3.12.3",
  "seed": 20260819
}


## 5. Freeze the experiment

**Experiment:** Run one stride-aware Triton copy on contiguous columns and on a stride-two view.

Inputs, output contract, timer, and target stay fixed across compared paths.


## 6. Inspect and execute the reviewed code

The next cell calls the shared reviewed kernel source, retains full samples in `metrics`, checks maximum error, and prints the bounded analysis.


In [2]:
metrics, analysis_en, analysis_zh = run_lesson(LESSON_NO)
print(json.dumps(metrics, indent=2, ensure_ascii=False))
print(analysis_en)


{
  "primary": 0.017311999574303627,
  "secondary": 0.017823999747633934,
  "max_abs_error": 0.0,
  "passed": true,
  "details": {
    "slowdown": 1.0295748721072218,
    "contiguous_stride": [
      1024,
      1
    ],
    "strided_stride": [
      2048,
      2
    ],
    "contiguous_samples_ms": [
      0.030880000442266464,
      0.02239999920129776,
      0.02067199908196926,
      0.018912000581622124,
      0.017343999817967415,
      0.018432000651955605,
      0.01913600042462349,
      0.018848000094294548,
      0.017376000061631203,
      0.01756799966096878,
      0.016736000776290894,
      0.01727999933063984,
      0.016992000862956047,
      0.016383999958634377,
      0.0161920003592968,
      0.016256000846624374,
      0.01651199907064438,
      0.016607999801635742,
      0.0163199994713068,
      0.016448000445961952
    ],
    "strided_samples_ms": [
      0.02534399926662445,
      0.019807999953627586,
      0.01740800030529499,
      0.018688000738620758,
   

## 7. Read the retained RTX 5090 result

**Environment:** NVIDIA GeForce RTX 5090; compute capability 12.0; PyTorch 2.13.0+cu130; CUDA runtime 13.0; Triton 3.7.1; Python 3.12.3.

| Measured field | Checked-in value |
|---|---:|
| Contiguous median | 0.0173 ms |
| Stride-two median | 0.0178 ms |
| Maximum absolute error | 0.000e+00 |
| Acceptance gate | true |


## 8. Explain without overclaiming

The same copy kernel took 0.0173 ms on contiguous columns and 0.0178 ms at stride two, a 1.03x ratio.

A named Triton or PyTorch CUDA path executed on the recorded GPU. The result applies to the printed shape, dtype, implementation, and software stack; internal hardware causes require profiler evidence.


## 9. Write the canonical artifact

The next cell stores the environment, full metrics, bilingual analysis, evidence label, and bounded conclusion.


In [3]:
artifact = Path("artifacts/rtx5090-result.json")
artifact.parent.mkdir(parents=True, exist_ok=True)
payload = {
    "lesson": LESSON_NO,
    "title": LESSON_TITLE,
    "environment": ENV,
    "evidence_label": 'native-backend',
    "metrics": metrics,
    "analysis_en": analysis_en,
    "analysis_zh": analysis_zh,
    "conclusion": 'Diagnose addresses first, then use profiler transactions to confirm the mechanism before redesigning the kernel.',
}
artifact.write_text(json.dumps(payload, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")
print(json.dumps(payload, indent=2, ensure_ascii=False))


{
  "lesson": 6,
  "title": "Diagnosing Coalescing with Strides",
  "environment": {
    "gpu": "NVIDIA GeForce RTX 5090",
    "compute_capability": "12.0",
    "torch": "2.13.0+cu130",
    "cuda_runtime": "13.0",
    "triton": "3.7.1",
    "triton_target": "GPUTarget(backend='cuda', arch=120, warp_size=32)",
    "python": "3.12.3",
    "seed": 20260819
  },
  "evidence_label": "native-backend",
  "metrics": {
    "primary": 0.017311999574303627,
    "secondary": 0.017823999747633934,
    "max_abs_error": 0.0,
    "passed": true,
    "details": {
      "slowdown": 1.0295748721072218,
      "contiguous_stride": [
        1024,
        1
      ],
      "strided_stride": [
        2048,
        2
      ],
      "contiguous_samples_ms": [
        0.030880000442266464,
        0.02239999920129776,
        0.02067199908196926,
        0.018912000581622124,
        0.017343999817967415,
        0.018432000651955605,
        0.01913600042462349,
        0.018848000094294548,
        0.01737600

## 10. Make the bounded decision

> Diagnose addresses first, then use profiler transactions to confirm the mechanism before redesigning the kernel.

**Failure analysis:** Adding shared memory before proving an access problem can increase work without fixing the actual address pattern.


## 11. Extend and review

Add an awkward shape and non-contiguous layout. Stop on correctness failure. See `README.md` for references and the full review checklist.
